# Phase 2c. Sample-level Immune Ecosystem Analysis

이 노트북의 목적은 C1QC-associated TAM이 우세한 샘플과 SPP1-associated TAM이 우세한 샘플은 T cell,Treg, DC 등 다른 면역세포 구성에서 체계적인 차이를 보이는지를 확인하는 것이다.

본 분석은 필요한 이유는 다음과 같다.

Phase 2a~2b는 TAM subtype 자체의 annotation과 pan-cancer 분포를 보였다.  
Phase 2c는 TAM state 차이가 샘플 수준의 immune ecosystem과 어떻게 연결되는가로 질문을 한 단계 올린다.  
이 분석이 있어야 Phase 3의 spatial co-localization 질문으로 자연스럽게 이어질 수 있다.  

## 분석 한계 (사전 명시)

- GSE127465 유효 샘플: 전체 26개 중 Unresolved 100% 샘플(8개) 제외 -> 최대 n=18  
- n = 18은 통계적 파워가 제한적이므로 결과는 탐색적 분석으로 해석한다.  
- Phase 1 cell type annotation은 대분류(T cell / B cell / Macrophage / Cancer cell) 기준이므로 Treg / NK / DC 등 세부 구분은 현재 annotation 수준에서 직접 추출 불가 -> 가능한 대분류 내 비율 비교로 진행하고 세부 분류를 별도 subclustering이 필요함을 명시한다.  

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from utils.paths import *
from utils.report import ph2_report_adata_state

sc.settings.set_figure_params(dpi=100, facecolor='white')
np.random.seed(42)

# 1. 데이터 로드 및 상태 확인

두 가지 데이터를 사용한다.

- `HUMAN_H5AD`: Phase 1 전체 cell type annotation (T cell / B cell / Macrophage / Cancer cell / Unknown)
- `MAC_TME_H5AD`: Phase 2a macrophage subset + `tam_subtype_final` annotation

In [2]:
# 전체 cell type annotation 데이터
adata = sc.read_h5ad(HUMAN_H5AD)
ph2_report_adata_state(adata, 'Phase 1 full data')
print(adata.obs['cell_type'].value_counts())
print('sample count:', adata.obs['sample'].nunique())

===== Phase 1 full data =====
shape: 44,860 cells x 2,000 genes
X type: <class 'numpy.ndarray'>
X dtype: float64
obs names unique: True
var names unique: True
obs columns: ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_counts', 'sample', 'leiden', 'cell_type']
obsm keys: ['X_pca', 'X_pca_harmony', 'X_umap']
obsp keys: ['connectivities', 'distances']
layers keys: ['counts']
uns keys: ['cell_type_colors', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap']

cell_type
Unknown        14937
T cell         12705
Macrophage      9567
B cell          4572
Cancer cell     3079
Name: count, dtype: int64
sample count: 26


In [3]:
# TAM subtype annotation 데이터
mac = sc.read_h5ad(MAC_TME_H5AD)
ph2_report_adata_state(mac, 'Phase 2a macrophage subset')

# tam_subtype_final 컬럼 확인
if 'tam_subtype_final' in mac.obs.columns:
    print(mac.obs['tam_subtype_final'].value_counts())
else:
    print('WARNING: tam_subtype_final 없음 — tam_subtype_merged 확인')
    print(mac.obs.columns.tolist())

===== Phase 2a macrophage subset =====
shape: 9,567 cells x 32,634 genes
X type: <class 'scipy.sparse._csr.csr_matrix'>
X dtype: float32
obs names unique: True
var names unique: True
obs columns: ['n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'mt_outlier', 'n_counts', 'sample', 'leiden', 'cell_type', 'leiden_7', 'leiden_8', 'leiden_9', 'leiden_10', 'leiden_12', 'leiden_15', 'leiden_r0.1', 'leiden_r0.2', 'leiden_r0.3', 'leiden_r0.4', 'leiden_r0.5', 'leiden_r0.6', 'leiden_r0.7', 'leiden_r0.8', 'leiden_r0.9', 'leiden_r1.0', 'C1QC_score', 'SPP1_score', 'Resting_C1QC_score', 'Activated_C1QC_score', 'ISG15_score', 'tam_subtype', 'tam_subtype_initial', 'tam_subtype_final', 'tam_subtype_merged']
obsm keys: ['X_pca', 'X_pca_harmony', 'X_umap']
obsp keys: ['co